# 자동차 감지 및 바운딩 박스 그리기

이 노트북에서는 YOLO 모델이 감지한 자동차 주위에 바운딩 박스를 그리는 방법을 학습합니다.

실습에 사용하는 `yolov8m.pt` 모델은 사전 학습된 YOLOv8 중간 사이즈 모델로, 자동차를 포함한 다양한 객체를 COCO 데이터셋 기준으로 감지할 수 있도록 훈련되어 있습니다.

COCO(Common Objects in Context)란 Microsoft가 공개한 대규모 오픈 이미지 데이터셋으로,  
객체 탐지(Object Detection), 세그멘테이션(Segmentation), 키포인트 추정(Keypoint Estimation), 이미지 캡셔닝(Image Captioning) 등의 작업을 컴퓨터 비전 모델에 학습시키기 위한 정보를 담고 있습니다. 

그럼, 이어서 실습을 진행합니다.

In [ ]:
# 이 실습을 위해 사전구성된 워크벤치 이미지를 사용하지 않았다면, 아래 줄의 주석을 해제하고 실행하여 필요한 패키지들을 설치할 수 있습니다.
# !pip install --no-cache-dir --no-dependencies -r requirements.txt

# YOLO 및 이미지 처리 패키지 불러오기
from ultralytics import YOLO  # YOLOv8 모델을 사용할 수 있는 라이브러리
from PIL import Image  # 이미지 로딩 및 표시를 위한 라이브러리

In [ ]:
# 이번 실습에서는 객체 감지를 위해 YOLOv8m 모델을 사용할 것입니다.

model = YOLO("yolov8m.pt")

In [ ]:
# 테스트 이미지에 대한 모델 예측 결과를 가져옵니다.

img = "images/carImage0.jpg"
results = model.predict(img)

In [ ]:
# YOLO는 하나의 이미지뿐만 아니라 이미지 배열을 입력받을 수 있으며, 이에 대한 결과 배열을 반환합니다.  
# 여기서는 하나의 이미지만 사용했기 때문에, 결과 배열 중 첫 번째 항목(results[0])만 추출합니다.

result = results[0]

In [ ]:
# YOLO의 예측 결과에서 감지된 객체(즉, 바운딩 박스)의 개수를 확인합니다.
# result.boxes는 이미지에서 감지된 모든 객체를 포함하는 리스트이며, len()을 통해 그 수를 셉니다.

len(result.boxes)

In [ ]:
# 바운딩 박스를 분석합니다.

box = result.boxes[0]  ## 첫 번째 바운딩 박스를 box 변수에 저장합니다.
print("Object type:", box.cls)  ## box.cls: 감지된 객체의 클래스 번호 (예: '2'는 자동차)
print("Coordinates:", box.xyxy)  ## box.xyxy: 바운딩 박스의 좌표 (좌상단 x,y ~ 우하단 x,y)
print("Probability:", box.conf)  ## box.conf: 신뢰도 (confidence)

In [ ]:
# 텐서(Tensor) 형태로 저장된 데이터를 리스트로 변환하고 변수에 저장합니다.
# .tolist() 및 .item()을 사용하여 좌표와 클래스 번호, 확률을 일반 숫자로 추출합니다.
# 추출된 값들을 출력해 확인합니다.

cords = box.xyxy[0].tolist()
class_id = box.cls[0].item()
conf = box.conf[0].item()
print("Object type:", class_id)
print("Coordinates:", cords)
print("Probability:", conf)

In [ ]:
# YOLOv8은 COCO 데이터셋에 기반하여 훈련되었으며, 이 데이터셋은 80개의 클래스(사람, 자동차, 자전거 등)를 정의합니다.
# 이 데이터셋에서 감지 대상 객체들은 클래스(class)로 분류되어 있으며, 클래스 번호와 객체 이름이 서로 매핑되어 있습니다. 
# 예 - {0: 'person', 1: 'bicycle', 2: 'car', ...}

# 클래스 번호와 객체 이름 조회
print(result.names)

클래스 번호 '2'는 '자동차(car)' 객체에 해당합니다.  
따라서 결과에 나온 바운딩 박스는 감지된 자동차를 나타냅니다.  
이제 이미지 위에 해당 박스를 그려보겠습니다!

In [ ]:
# 먼저, 바운딩 박스의 좌표들을 리스트에 저장하고, 반올림 처리합니다.  
# 그런 다음, result.names 딕셔너리를 이용해 객체의 클래스 ID에 해당하는 이름을 가져옵니다.

cords = box.xyxy[0].tolist()  ## 바운딩 박스의 좌표를 리스트로 변환
cords = [round(x) for x in cords]  ## 각 좌표 값을 반올림
class_id = result.names[box.cls[0].item()]  ## 객체 이름을 클래스 ID에 매핑
conf = round(box.conf[0].item(), 2)  ## 신뢰도(confidence) 반올림
print("Object type:", class_id)
print("Coordinates:", cords)
print("Probability:", conf)

In [ ]:
# 이제 감지된 모든 객체(바운딩 박스)에 대해, 관련 정보를 추출하는 작업을 반복적으로 수행하도록 하겠습니다.

for box in result.boxes:
  class_id = result.names[box.cls[0].item()]
  cords = box.xyxy[0].tolist()
  cords = [round(x) for x in cords]
  conf = round(box.conf[0].item(), 2)
  print("Object type:", class_id)
  print("Coordinates:", cords)
  print("Probability:", conf)
  print("---")

In [ ]:
# 이미지 위에 박스를 그리고, 감지된 클래스의 이름과 감지 신뢰도(모델이 개별 예측 결과에 대해 얼마나 확신하는지를 나타냄)을 함께 표시합니다.

## result.plot()은 감지된 객체 위에 바운딩 박스를 그린 이미지를 반환
## OpenCV 색상 채널(BGR → RGB) 순서 맞추기 위해 [:, :, ::-1] 사용
## Image.fromarray()로 PIL 이미지 객체로 변환하여 Jupyter에서 표시 가능

Image.fromarray(result.plot()[:, :, ::-1])

이제 이전 노트북에서 테스트했던 여러 대의 자동차가 포함된 이미지(carImage4.jpg)로 돌아가서, YOLO가 이미지 내의 모든 '자동차(car)'를 실제로 감지할 수 있는지 확인해보겠습니다.

In [ ]:
# 아래 코드는 이전 셀들에서 사용한 내용과 동일하지만, 한 번에 실행되도록 구성된 버전입니다.

results = model.predict("images/carImage4.jpg")

result = results[0]

for box in result.boxes:
  class_id = result.names[box.cls[0].item()]
  cords = box.xyxy[0].tolist()
  cords = [round(x) for x in cords]
  conf = round(box.conf[0].item(), 2)
  print("Object type:", class_id)
  print("Coordinates:", cords)
  print("Probability:", conf)
  print("---")

Image.fromarray(result.plot()[:,:,::-1])

YOLO 모델이 이미지의 '멀리 뒤쪽'에 있는 일부 자동차는 감지하지 못한 것으로 보입니다.  
하지만 전반적으로, 이 모델은 이미지에 포함된 여러 대의 자동차를 잘 감지해냈습니다.  
그리고 더 중요한 점은, 감지된 자동차들이 모두 '바운딩 박스'로 잘 둘러싸여 있다는 것입니다.

이제 YOLO 모델이 감지한 자동차에 바운딩 박스를 정확히 그릴 수 있게 되었으므로,  
다음 단계로는 자동차 '사고(crash)'를 감지할 수 있도록 YOLO 모델을 재학습시킬 수 있습니다.

**노트북 `04-03-model-retraining.ipynb`를 열어 주세요.**